#### Strategy Outline

Bias for commercial net long > 80 and <20 for net short
When bias is long and RSI is <30 go long, go short when bias is short and rsi is  over 70 go short

Exit: RSI @60 or 20 day limit

Risk management: 2 ATR stop, 3 ATR target, 1% risk per trade

In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

with open('cot_data.json', 'r') as f:
    data__raw = json.load(f)

df = pd.DataFrame(data__raw)
df["Date"] = pd.to_datetime(df["Date"], unit='ms')  # Convert from milliseconds


#### Prepare data for analysis

In [3]:
def prepare_strategy_data(df, market_name):
    """
    Prepare strategy data for a specific market.
    Combines daily OHLC price data with weekly COT data.
    
    Returns DataFrame with: Date, Open, High, Low, Close, RSI, Commercial_Index, OI
    """
    market_data = df[df['Market'] == market_name].copy()

    cot_weekly = market_data[market_data['data_type'] == 'weekly_cot'].copy()
    price_daily = market_data[market_data['data_type'] == 'daily_price'].copy()

    if price_daily.empty:
        print(f"⚠ No daily price data for {market_name}")
        return pd.DataFrame()  # Return empty DataFrame instead of COT-only data

    # Include OHLC columns for ATR calculation
    price_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'RSI']
    
    # Check which columns actually exist
    available_cols = [col for col in price_cols if col in price_daily.columns]
    missing_cols = [col for col in price_cols if col not in price_daily.columns]
    
    if missing_cols:
        print(f"⚠ {market_name}: Missing columns: {missing_cols}")
    
    strategy_data = price_daily[available_cols].copy()

    # Merge weekly COT data directly, then forward fill
    cot_cols = ['Net Commercial Position', 'OI', 'Commercial_Index']
    cot_for_merge = cot_weekly[['Date'] + cot_cols].copy()
    
    strategy_data = pd.merge(strategy_data, cot_for_merge, on='Date', how='left')
    
    # Forward fill COT values to fill gaps between weekly reports
    strategy_data[cot_cols] = strategy_data[cot_cols].ffill()
    
    # Remove rows with missing critical data (need Close and Commercial_Index at minimum)
    strategy_data = strategy_data.dropna(subset=['Close', 'Commercial_Index'])
    
    # Sort by date and reset index
    strategy_data = strategy_data.sort_values('Date').reset_index(drop=True)
    
    print(f"✓ {market_name}: {len(strategy_data)} rows, columns: {list(strategy_data.columns)}")
    
    return strategy_data 



In [4]:
# Test the function with a market
test_market = "GOLD - COMMODITY EXCHANGE INC."
test_data = prepare_strategy_data(df, test_market)

# Display first few rows to verify OHLC is included
print(f"\nFirst 5 rows:")
print(test_data.head())
print(f"\nLast 5 rows:")
print(test_data.tail())

✓ GOLD - COMMODITY EXCHANGE INC.: 502 rows, columns: ['Date', 'Open', 'High', 'Low', 'Close', 'RSI', 'Net Commercial Position', 'OI', 'Commercial_Index']

First 5 rows:
        Date         Open         High          Low        Close  RSI  \
0 2024-01-02  2063.500000  2073.699951  2057.100098  2064.399902  NaN   
1 2024-01-03  2034.199951  2044.000000  2034.199951  2034.199951  NaN   
2 2024-01-04  2041.599976  2044.500000  2038.000000  2042.300049  NaN   
3 2024-01-05  2044.500000  2048.100098  2042.400024  2042.400024  NaN   
4 2024-01-08  2019.099976  2033.699951  2019.099976  2026.599976  NaN   

   Net Commercial Position        OI  Commercial_Index  
0                -235478.0  500364.0         55.818686  
1                -235478.0  500364.0         55.818686  
2                -235478.0  500364.0         55.818686  
3                -235478.0  500364.0         55.818686  
4                -235478.0  500364.0         55.818686  

Last 5 rows:
          Date         Open         

### Technical Indicators

In [5]:
def calculate_atr(data, period=10):

    df = data.copy()
    
    # Validate required columns exist
    required = ['High', 'Low', 'Close']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for ATR: {missing}")
    
    # Calculate True Range components
    df['prev_close'] = df['Close'].shift(1)
    
    df['tr1'] = df['High'] - df['Low']                    # Current bar range
    df['tr2'] = abs(df['High'] - df['prev_close'])        # High vs previous close
    df['tr3'] = abs(df['Low'] - df['prev_close'])         # Low vs previous close
    
    # True Range is the maximum of the three
    df['TR'] = df[['tr1', 'tr2', 'tr3']].max(axis=1)
    
    # ATR is the rolling mean of True Range
    df['ATR'] = df['TR'].rolling(window=period).mean()
    
    # Clean up temporary columns
    df.drop(columns=['prev_close', 'tr1', 'tr2', 'tr3', 'TR'], inplace=True)
    
    # Report ATR coverage
    valid_atr = df['ATR'].notna().sum()
    total_rows = len(df)
    print(f"✓ ATR calculated: {valid_atr}/{total_rows} rows have valid ATR (first {period} rows are NaN)")
    
    return df


# Test ATR calculation
test_data_with_atr = calculate_atr(test_data.copy())
print(f"\nATR sample (last 5 rows):")
print(test_data_with_atr[['Date', 'Close', 'High', 'Low', 'ATR']].tail())


✓ ATR calculated: 493/502 rows have valid ATR (first 10 rows are NaN)

ATR sample (last 5 rows):
          Date        Close         High          Low        ATR
497 2025-12-22  4444.600098  4447.600098  4371.100098  56.050000
498 2025-12-23  4482.799805  4503.799805  4450.399902  57.769971
499 2025-12-24  4480.600098  4503.399902  4468.399902  56.179980
500 2025-12-26  4529.100098  4556.299805  4502.000000  54.699951
501 2025-12-29  4325.100098  4379.000000  4325.100098  65.599951


In [6]:
def generate_signals(data, commercial_long_threshold=80, commercial_short_threshold=20, 
                     rsi_oversold=30, rsi_overbought=70):
    """
    Generate trading signals based on COT Commercial Index and RSI.
    
    Strategy Rules:
    - LONG:  Commercial_Index >= 80 AND RSI < 30
    - SHORT: Commercial_Index <= 20 AND RSI > 70
    
    Args:
        data: DataFrame with 'Commercial_Index' and 'RSI' columns
        commercial_long_threshold: Commercial Index level for long bias (default 80)
        commercial_short_threshold: Commercial Index level for short bias (default 20)
        rsi_oversold: RSI level for oversold (default 30)
        rsi_overbought: RSI level for overbought (default 70)
    
    Returns:
        DataFrame with 'signal' column added (1=Long, -1=Short, 0=None)
    """
    df = data.copy()
    
    # Validate required columns
    required = ['Commercial_Index', 'RSI']
    missing = [col for col in required if col not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns for signals: {missing}")
    
    # Initialize signal column
    df['signal'] = 0
    
    # Long signal: Commercial bullish (>=80) AND RSI oversold (<30)
    long_condition = (df['Commercial_Index'] >= commercial_long_threshold) & (df['RSI'] < rsi_oversold)
    df.loc[long_condition, 'signal'] = 1
    
    # Short signal: Commercial bearish (<=20) AND RSI overbought (>70)
    short_condition = (df['Commercial_Index'] <= commercial_short_threshold) & (df['RSI'] > rsi_overbought)
    df.loc[short_condition, 'signal'] = -1
    
    # Count signals
    long_signals = (df['signal'] == 1).sum()
    short_signals = (df['signal'] == -1).sum()
    total_rows = len(df)
    
    print(f"✓ Signals generated:")
    print(f"   Long signals:  {long_signals} (Commercial >= {commercial_long_threshold} AND RSI < {rsi_oversold})")
    print(f"   Short signals: {short_signals} (Commercial <= {commercial_short_threshold} AND RSI > {rsi_overbought})")
    print(f"   No signal:     {total_rows - long_signals - short_signals}")
    
    return df


# Test signal generation
test_data_with_signals = generate_signals(test_data_with_atr.copy())

# Show rows where we have signals
signal_rows = test_data_with_signals[test_data_with_signals['signal'] != 0]
if not signal_rows.empty:
    print(f"\nSignal dates:")
    print(signal_rows[['Date', 'Close', 'RSI', 'Commercial_Index', 'signal']].to_string())
else:
    print("\n⚠ No signals found for this market with default thresholds")


✓ Signals generated:
   Long signals:  2 (Commercial >= 80 AND RSI < 30)
   Short signals: 50 (Commercial <= 20 AND RSI > 70)
   No signal:     450

Signal dates:
          Date        Close         RSI  Commercial_Index  signal
29  2024-02-13  1992.900024   20.342687        100.000000       1
30  2024-02-14  1990.300049   19.269853        100.000000       1
134 2024-07-16  2462.399902   99.002992         16.281610      -1
135 2024-07-17  2454.800049   92.029786         16.281610      -1
136 2024-07-18  2451.800049   89.540249         16.281610      -1
159 2024-08-20  2511.300049   72.781686         12.424622      -1
160 2024-08-21  2508.399902   70.736369         12.424622      -1
175 2024-09-12  2551.199951   79.708618         17.569594      -1
176 2024-09-13  2581.300049   84.828262         17.569594      -1
177 2024-09-16  2580.399902   99.749748         17.569594      -1
178 2024-09-17  2564.300049   83.020994          2.452256      -1
179 2024-09-18  2570.699951   84.082169      

In [7]:
class COTRSIBacktester:
    """
    Backtester for COT + RSI trading strategy.
    
    Key Features:
    - Ignores duplicate signals while position is open (no stacking)
    - Exits on RSI cross (60), 20-day limit, stop loss, or take profit
    - Position sizing based on 1% risk per trade
    - Tracks all trades with entry/exit details
    """
    
    def __init__(self, initial_capital=30000, risk_per_trade=0.01, 
                 atr_stop_mult=2, atr_target_mult=3, max_hold_days=20, rsi_exit=60):
        """
        Initialize backtester.
        
        Args:
            initial_capital: Starting capital (default $30,000)
            risk_per_trade: Fraction of capital to risk per trade (default 1%)
            atr_stop_mult: ATR multiplier for stop loss (default 2x)
            atr_target_mult: ATR multiplier for take profit (default 3x)
            max_hold_days: Maximum days to hold a trade (default 20)
            rsi_exit: RSI level to exit trades (default 60)
        """
        self.initial_capital = initial_capital
        self.risk_per_trade = risk_per_trade
        self.atr_stop_mult = atr_stop_mult
        self.atr_target_mult = atr_target_mult
        self.max_hold_days = max_hold_days
        self.rsi_exit = rsi_exit
        
        # State tracking
        self.capital = initial_capital
        self.trades = []
        self.equity_curve = []
        
    def calculate_position_size(self, entry_price, atr):
        """
        Calculate position size based on 1% risk rule.
        
        Formula:
        - Stop distance = 2 x ATR
        - Risk amount = capital x risk_per_trade
        - Position value = risk_amount / (stop_distance / entry_price)
        - Position units = position_value / entry_price (rounded to integer)
        """
        if pd.isna(atr) or atr <= 0:
            return 0, f"Invalid ATR: {atr}"
        
        stop_distance = self.atr_stop_mult * atr
        stop_price = entry_price - stop_distance  # For long; adjust for short in backtest
        
        risk_amount = self.capital * self.risk_per_trade
        risk_per_unit = stop_distance
        
        if risk_per_unit <= 0:
            return 0, f"Invalid risk per unit: {risk_per_unit}"
        
        units = int(risk_amount / risk_per_unit)
        
        if units <= 0:
            return 0, f"Position size too small: {units}"
            
        return units, None
    
    def backtest(self, data, market_name="Unknown"):
        """
        Run backtest on prepared data.
        
        Args:
            data: DataFrame with Date, Open, High, Low, Close, RSI, ATR, signal columns
            market_name: Name of market for reporting
            
        Returns:
            dict with trades list and equity curve
        """
        df = data.copy().reset_index(drop=True)
        
        # Validate required columns
        required = ['Date', 'Open', 'Close', 'RSI', 'ATR', 'signal']
        missing = [col for col in required if col not in df.columns]
        if missing:
            raise ValueError(f"Missing columns: {missing}")
        
        # State variables
        in_position = False
        position_direction = 0  # 1=long, -1=short
        entry_price = 0
        entry_date = None
        entry_idx = 0
        stop_loss = 0
        take_profit = 0
        units = 0
        
        trades = []
        equity = [self.initial_capital]
        current_capital = self.initial_capital
        
        for i in range(len(df)):
            row = df.iloc[i]
            date = row['Date']
            open_price = row['Open']
            high = row['High'] if 'High' in df.columns else row['Close']
            low = row['Low'] if 'Low' in df.columns else row['Close']
            close = row['Close']
            rsi = row['RSI']
            atr = row['ATR']
            signal = row['signal']
            
            # Skip if missing critical data
            if pd.isna(close) or pd.isna(rsi):
                equity.append(current_capital)
                continue
            
            # ===== CHECK EXITS FIRST =====
            if in_position:
                days_held = i - entry_idx
                exit_reason = None
                exit_price = None
                
                if position_direction == 1:  # Long position
                    # Check stop loss (hit during day)
                    if low <= stop_loss:
                        exit_reason = "Stop Loss"
                        exit_price = stop_loss
                    # Check take profit (hit during day)
                    elif high >= take_profit:
                        exit_reason = "Take Profit"
                        exit_price = take_profit
                    # Check RSI exit (close >= 60)
                    elif rsi >= self.rsi_exit:
                        exit_reason = f"RSI Exit ({rsi:.1f})"
                        exit_price = close
                    # Check max hold days
                    elif days_held >= self.max_hold_days:
                        exit_reason = f"Max Hold ({days_held} days)"
                        exit_price = close
                        
                else:  # Short position
                    # Check stop loss (hit during day)
                    if high >= stop_loss:
                        exit_reason = "Stop Loss"
                        exit_price = stop_loss
                    # Check take profit (hit during day)
                    elif low <= take_profit:
                        exit_reason = "Take Profit"
                        exit_price = take_profit
                    # Check RSI exit (close <= 60 for shorts... wait, strategy says RSI @60)
                    # For shorts, we exit when RSI drops back to 60 (from overbought)
                    elif rsi <= self.rsi_exit:
                        exit_reason = f"RSI Exit ({rsi:.1f})"
                        exit_price = close
                    # Check max hold days
                    elif days_held >= self.max_hold_days:
                        exit_reason = f"Max Hold ({days_held} days)"
                        exit_price = close
                
                # Execute exit if triggered
                if exit_reason:
                    if position_direction == 1:
                        pnl = (exit_price - entry_price) * units
                    else:
                        pnl = (entry_price - exit_price) * units
                    
                    pnl_pct = (pnl / (entry_price * units)) * 100 if units > 0 else 0
                    current_capital += pnl
                    
                    trades.append({
                        'market': market_name,
                        'entry_date': entry_date,
                        'exit_date': date,
                        'direction': 'Long' if position_direction == 1 else 'Short',
                        'entry_price': entry_price,
                        'exit_price': exit_price,
                        'units': units,
                        'pnl': pnl,
                        'pnl_pct': pnl_pct,
                        'exit_reason': exit_reason,
                        'days_held': days_held
                    })
                    
                    in_position = False
                    position_direction = 0
            
            # ===== CHECK ENTRIES (only if not in position) =====
            if not in_position and signal != 0 and not pd.isna(atr):
                # Enter at next day's open (use current close as proxy)
                entry_price = close
                entry_date = date
                entry_idx = i
                position_direction = signal
                
                # Calculate position size
                units, error = self.calculate_position_size(entry_price, atr)
                
                if error:
                    # Skip this signal - can't size position
                    continue
                
                # Set stop loss and take profit
                if signal == 1:  # Long
                    stop_loss = entry_price - (self.atr_stop_mult * atr)
                    take_profit = entry_price + (self.atr_target_mult * atr)
                else:  # Short
                    stop_loss = entry_price + (self.atr_stop_mult * atr)
                    take_profit = entry_price - (self.atr_target_mult * atr)
                
                in_position = True
            
            equity.append(current_capital)
        
        # Close any open position at end
        if in_position:
            final_close = df.iloc[-1]['Close']
            final_date = df.iloc[-1]['Date']
            days_held = len(df) - 1 - entry_idx
            
            if position_direction == 1:
                pnl = (final_close - entry_price) * units
            else:
                pnl = (entry_price - final_close) * units
            
            pnl_pct = (pnl / (entry_price * units)) * 100 if units > 0 else 0
            current_capital += pnl
            
            trades.append({
                'market': market_name,
                'entry_date': entry_date,
                'exit_date': final_date,
                'direction': 'Long' if position_direction == 1 else 'Short',
                'entry_price': entry_price,
                'exit_price': final_close,
                'units': units,
                'pnl': pnl,
                'pnl_pct': pnl_pct,
                'exit_reason': 'End of Data',
                'days_held': days_held
            })
            equity.append(current_capital)
        
        self.trades = trades
        self.equity_curve = equity
        self.capital = current_capital
        
        return {
            'trades': pd.DataFrame(trades),
            'equity_curve': equity,
            'final_capital': current_capital,
            'total_return': (current_capital - self.initial_capital) / self.initial_capital * 100
        }


# Test the backtester
print("=" * 80)
print("BACKTESTER TEST")
print("=" * 80)

backtester = COTRSIBacktester(
    initial_capital=100000,
    risk_per_trade=0.01,
    atr_stop_mult=2,
    atr_target_mult=3,
    max_hold_days=20,
    rsi_exit=60
)

results = backtester.backtest(test_data_with_signals, market_name=test_market)

print(f"\n📊 Results for {test_market}:")
print(f"   Initial Capital: ${backtester.initial_capital:,.2f}")
print(f"   Final Capital:   ${results['final_capital']:,.2f}")
print(f"   Total Return:    {results['total_return']:.2f}%")
print(f"   Total Trades:    {len(results['trades'])}")

if not results['trades'].empty:
    print(f"\n📝 Trade Summary:")
    print(results['trades'][['entry_date', 'exit_date', 'direction', 'entry_price', 
                             'exit_price', 'pnl', 'exit_reason']].to_string())


BACKTESTER TEST

📊 Results for GOLD - COMMODITY EXCHANGE INC.:
   Initial Capital: $100,000.00
   Final Capital:   $96,977.61
   Total Return:    -3.02%
   Total Trades:    15

📝 Trade Summary:
   entry_date  exit_date direction  entry_price   exit_price          pnl      exit_reason
0  2024-02-13 2024-02-23      Long  1992.900024  2038.599976  1051.098877  RSI Exit (79.4)
1  2024-07-16 2024-07-19     Short  2462.399902  2395.500000  1204.198242  RSI Exit (58.2)
2  2024-08-20 2024-08-22     Short  2511.300049  2478.899902   550.802490  RSI Exit (55.0)
3  2024-09-12 2024-09-20     Short  2551.199951  2598.879883  -953.598633        Stop Loss
4  2024-09-20 2024-09-25     Short  2619.899902  2663.619922  -961.840430        Stop Loss
5  2024-09-25 2024-10-04     Short  2659.199951  2645.800049   281.397949  RSI Exit (51.2)
6  2024-10-17 2024-10-21     Short  2691.000000  2733.179980  -970.139551        Stop Loss
7  2024-10-21 2024-10-29     Short  2723.100098  2766.660107  -958.320215     

In [8]:
def calculate_performance_metrics(trades_df, equity_curve, initial_capital=100000):
    """
    Calculate comprehensive performance metrics for a backtest.
    
    Args:
        trades_df: DataFrame with trade records (must have 'pnl' column)
        equity_curve: List of equity values over time
        initial_capital: Starting capital
    
    Returns:
        dict with performance metrics
    """
    metrics = {}
    
    if trades_df.empty:
        return {
            'total_trades': 0,
            'win_rate': 0,
            'profit_factor': 0,
            'total_return_pct': 0,
            'cagr': 0,
            'max_drawdown_pct': 0,
            'sharpe_ratio': 0,
            'avg_win': 0,
            'avg_loss': 0,
            'largest_win': 0,
            'largest_loss': 0,
            'avg_days_held': 0
        }
    
    # Basic trade stats
    total_trades = len(trades_df)
    winning_trades = trades_df[trades_df['pnl'] > 0]
    losing_trades = trades_df[trades_df['pnl'] < 0]
    
    win_count = len(winning_trades)
    loss_count = len(losing_trades)
    
    metrics['total_trades'] = total_trades
    metrics['winning_trades'] = win_count
    metrics['losing_trades'] = loss_count
    metrics['win_rate'] = (win_count / total_trades * 100) if total_trades > 0 else 0
    
    # P&L stats
    total_profit = winning_trades['pnl'].sum() if not winning_trades.empty else 0
    total_loss = abs(losing_trades['pnl'].sum()) if not losing_trades.empty else 0
    
    metrics['gross_profit'] = total_profit
    metrics['gross_loss'] = total_loss
    metrics['net_profit'] = total_profit - total_loss
    metrics['profit_factor'] = (total_profit / total_loss) if total_loss > 0 else float('inf')
    
    # Average trade stats
    metrics['avg_win'] = winning_trades['pnl'].mean() if not winning_trades.empty else 0
    metrics['avg_loss'] = losing_trades['pnl'].mean() if not losing_trades.empty else 0
    metrics['avg_trade'] = trades_df['pnl'].mean()
    
    # Largest trades
    metrics['largest_win'] = winning_trades['pnl'].max() if not winning_trades.empty else 0
    metrics['largest_loss'] = losing_trades['pnl'].min() if not losing_trades.empty else 0
    
    # Days held
    if 'days_held' in trades_df.columns:
        metrics['avg_days_held'] = trades_df['days_held'].mean()
    
    # Return metrics
    final_capital = equity_curve[-1] if equity_curve else initial_capital
    total_return = (final_capital - initial_capital) / initial_capital
    metrics['total_return_pct'] = total_return * 100
    
    # CAGR (Compound Annual Growth Rate)
    # Assume ~252 trading days per year
    if 'entry_date' in trades_df.columns and 'exit_date' in trades_df.columns:
        first_date = trades_df['entry_date'].min()
        last_date = trades_df['exit_date'].max()
        days_traded = (last_date - first_date).days
        years = days_traded / 365.25 if days_traded > 0 else 1
    else:
        years = len(equity_curve) / 252  # Approximate
    
    if years > 0 and final_capital > 0:
        metrics['cagr'] = ((final_capital / initial_capital) ** (1 / years) - 1) * 100
    else:
        metrics['cagr'] = 0
    
    # Max Drawdown
    equity_series = pd.Series(equity_curve)
    rolling_max = equity_series.cummax()
    drawdown = (equity_series - rolling_max) / rolling_max
    metrics['max_drawdown_pct'] = abs(drawdown.min()) * 100
    
    # Sharpe Ratio (simplified - using trade returns)
    if len(trades_df) > 1:
        trade_returns = trades_df['pnl_pct'] / 100 if 'pnl_pct' in trades_df.columns else trades_df['pnl'] / initial_capital
        avg_return = trade_returns.mean()
        std_return = trade_returns.std()
        # Annualize assuming ~20 trades per year (rough estimate)
        trades_per_year = 252 / trades_df['days_held'].mean() if 'days_held' in trades_df.columns else 20
        metrics['sharpe_ratio'] = (avg_return * trades_per_year) / (std_return * np.sqrt(trades_per_year)) if std_return > 0 else 0
    else:
        metrics['sharpe_ratio'] = 0
    
    return metrics


# Test performance metrics
metrics = calculate_performance_metrics(results['trades'], results['equity_curve'], backtester.initial_capital)

print("=" * 80)
print("PERFORMANCE METRICS")
print("=" * 80)
print(f"\n📊 Trade Statistics:")
print(f"   Total Trades:    {metrics['total_trades']}")
print(f"   Winning Trades:  {metrics['winning_trades']}")
print(f"   Losing Trades:   {metrics['losing_trades']}")
print(f"   Win Rate:        {metrics['win_rate']:.1f}%")

print(f"\n💰 Profit/Loss:")
print(f"   Gross Profit:    ${metrics['gross_profit']:,.2f}")
print(f"   Gross Loss:      ${metrics['gross_loss']:,.2f}")
print(f"   Net Profit:      ${metrics['net_profit']:,.2f}")
print(f"   Profit Factor:   {metrics['profit_factor']:.2f}")

print(f"\n📈 Returns:")
print(f"   Total Return:    {metrics['total_return_pct']:.2f}%")
print(f"   CAGR:            {metrics['cagr']:.2f}%")
print(f"   Max Drawdown:    {metrics['max_drawdown_pct']:.2f}%")
print(f"   Sharpe Ratio:    {metrics['sharpe_ratio']:.2f}")

print(f"\n📝 Trade Details:")
print(f"   Avg Win:         ${metrics['avg_win']:,.2f}")
print(f"   Avg Loss:        ${metrics['avg_loss']:,.2f}")
print(f"   Largest Win:     ${metrics['largest_win']:,.2f}")
print(f"   Largest Loss:    ${metrics['largest_loss']:,.2f}")
print(f"   Avg Days Held:   {metrics['avg_days_held']:.1f}")


PERFORMANCE METRICS

📊 Trade Statistics:
   Total Trades:    15
   Winning Trades:  7
   Losing Trades:   8
   Win Rate:        46.7%

💰 Profit/Loss:
   Gross Profit:    $4,714.19
   Gross Loss:      $7,736.58
   Net Profit:      $-3,022.39
   Profit Factor:   0.61

📈 Returns:
   Total Return:    -3.02%
   CAGR:            -1.90%
   Max Drawdown:    6.58%
   Sharpe Ratio:    -1.49

📝 Trade Details:
   Avg Win:         $673.46
   Avg Loss:        $-967.07
   Largest Win:     $1,204.20
   Largest Loss:    $-991.42
   Avg Days Held:   4.1
